# Business Entity Resolution — Exploratory Data Analysis

Goal: understand the data well enough to design **normalisation → blocking → pair model → assignment**.

Sections
1. Load & basic overview (sizes, nulls, countries, train vs test)
2. Ground truth structure (matches per entity, singletons, S2/S3 mix, exclusivity, distractors)
3. Text characteristics / noise patterns per source × country
4. Name & address vocabulary (legal suffixes, state formats)
5. Example matched clusters
6. Similarity signal: matched pairs vs random pairs
7. Blocking preview: how many true pairs share cheap keys
8. France (test-only country)

In [ ]:
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from IPython.display import display
from rapidfuzz import fuzz, process, utils

DATA = Path("dataset/student_resource/dataset")
SEED = 42

pl.Config.set_tbl_rows(40)
pl.Config.set_tbl_cols(30)
pl.Config.set_fmt_str_lengths(100)
pl.Config.set_tbl_width_chars(250)
plt.rcParams["figure.figsize"] = (11, 4)

## 1. Load & basic overview

In [ ]:
def load(split, name):
    """Read one challenge TSV with every column kept as a string."""
    return pl.read_csv(DATA / split / f"{split}_{name}.tsv", separator="\t", infer_schema=False)


t0 = time.time()
tr = {f"S{i}": load("train", f"source{i}") for i in (1, 2, 3)}
te = {f"S{i}": load("test", f"source{i}") for i in (1, 2, 3)}
gt = load("train", "ground_truth")
print(f"loaded in {time.time() - t0:.0f}s")

for split, d in [("train", tr), ("test", te)]:
    for k, df in d.items():
        print(f"{split} {k}: {df.shape}, unique ids={df['entity_id'].n_unique() == len(df)}, "
              f"prefix ok={df['entity_id'].str.starts_with(k + '-').all()}")
print("ground truth:", gt.shape)
tr["S1"].head(3)

In [ ]:
def overview(df, label):
    """Row count and null rates per country for one file."""
    return (
        df.group_by("country")
        .agg(
            rows=pl.len(),
            null_name=pl.col("business_name").is_null().mean().round(4),
            null_addr=pl.col("business_address").is_null().mean().round(4),
        )
        .with_columns(file=pl.lit(label))
    )


ov = pl.concat(
    [overview(df, f"train {k}") for k, df in tr.items()] + [overview(df, f"test {k}") for k, df in te.items()]
)
ov = ov.with_columns(share=(pl.col("rows") / pl.col("rows").sum().over("file")).round(3))
ov.select("file", "country", "rows", "share", "null_name", "null_addr").sort("file", "country")

In [ ]:
# Country mix: train vs test, per source
piv = ov.pivot(on="country", index="file", values="share").sort("file").fill_null(0)
display(piv)
left = np.zeros(piv.height)
for c in piv.columns[1:]:
    plt.barh(piv["file"], piv[c], left=left, label=c)
    left += piv[c].to_numpy()
plt.title("Country share per file"); plt.xlabel("share of rows"); plt.legend(); plt.tight_layout(); plt.show()

## 2. Ground truth structure

In [ ]:
gt = gt.with_columns(ids=pl.col("matched_entity_ids").str.split(","))
gt = gt.with_columns(
    n_match=pl.col("ids").list.len().fill_null(0),
    n_s2=pl.col("ids").list.eval(pl.element().str.starts_with("S2-")).list.sum().fill_null(0),
    n_s3=pl.col("ids").list.eval(pl.element().str.starts_with("S3-")).list.sum().fill_null(0),
).join(tr["S1"].select("entity_id", "country"), left_on="source1_entity_id", right_on="entity_id", how="left")

print("every GT S1 id is in train S1:", gt["country"].null_count() == 0)
print("every train S1 id is in GT   :", tr["S1"]["entity_id"].is_in(gt["source1_entity_id"].implode()).all())

gt.group_by("country").agg(
    entities=pl.len(),
    singleton_rate=(pl.col("n_match") == 0).mean().round(4),
    mean_matches=pl.col("n_match").mean().round(2),
    median_matches=pl.col("n_match").median(),
    max_matches=pl.col("n_match").max(),
    mean_s2=pl.col("n_s2").mean().round(2),
    mean_s3=pl.col("n_s3").mean().round(2),
    has_s2_and_s3=((pl.col("n_s2") > 0) & (pl.col("n_s3") > 0)).mean().round(3),
    only_s2=((pl.col("n_s2") > 0) & (pl.col("n_s3") == 0)).mean().round(3),
    only_s3=((pl.col("n_s2") == 0) & (pl.col("n_s3") > 0)).mean().round(3),
).sort("country")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ["n_match", "n_s2", "n_s3"]):
    for c in sorted(gt["country"].unique()):
        vc = gt.filter(pl.col("country") == c)[col].clip(upper_bound=12).value_counts(normalize=True).sort(col)
        ax.plot(vc[col], vc["proportion"], marker="o", label=c)
    ax.set_title(f"{col} per S1 entity (clipped at 12)"); ax.set_xlabel(col); ax.legend()
axes[0].set_ylabel("share of S1 entities")
plt.tight_layout(); plt.show()

**Exclusivity & distractors.** Does each S2/S3 record belong to at most one S1 entity? And what fraction of S2/S3 records match nothing (distractors the model must reject)?

In [ ]:
pairs = gt.select(s1_id="source1_entity_id", mid="ids").explode("mid", empty_as_null=True).drop_nulls("mid")
other_tr = pl.concat([tr["S2"], tr["S3"]]).with_columns(src=pl.col("entity_id").str.slice(0, 2))

print(f"true pairs: {len(pairs):,}")
print("S2/S3 ids appearing under >1 S1 :", pairs["mid"].is_duplicated().sum())
print("GT ids missing from S2/S3 files :", (~pairs["mid"].is_in(other_tr["entity_id"].implode())).sum())

other_tr = other_tr.with_columns(matched=pl.col("entity_id").is_in(pairs["mid"].implode()))
other_tr.group_by("src", "country").agg(
    rows=pl.len(), matched_rate=pl.col("matched").mean().round(3), distractor_rate=(~pl.col("matched")).mean().round(3)
).sort("src", "country")

In [ ]:
# Attach names/addresses to every true pair (used in the rest of the notebook)
s1a = tr["S1"].rename({"entity_id": "s1_id", "business_name": "name1", "business_address": "addr1", "country": "country1"})
oa = other_tr.select(
    mid="entity_id", name2="business_name", addr2="business_address", country2="country", src="src"
)
P = pairs.join(s1a, on="s1_id").join(oa, on="mid")
print(f"joined pairs: {len(P):,}")
print("pairs whose countries differ:", (P["country1"] != P["country2"]).sum())
P.group_by("country1", "country2").len().sort("len", descending=True)

## 3. Text characteristics / noise patterns

In [ ]:
INDIC = r"[\p{Bengali}\p{Gujarati}\p{Gurmukhi}\p{Kannada}\p{Malayalam}\p{Oriya}\p{Tamil}\p{Telugu}]"
NAME_FLAGS = {
    "devanagari": r"\p{Devanagari}",
    "other_indic": INDIC,
    "accented": r"[À-ÖØ-öø-ÿ]",
    "junk_prefix": r"^[^\p{L}\p{N}]",
    "domain_or_handle": r"(?i)(\.(com|net|org|in|co|fr|biz|info|us)\b|^@|www\.)",
    "brackets": r"[\(\[]",
    "dba": r"(?i)\b(d/?b/?a|doing business as|t/a|trading as)\b",
    "double_space": r"\s{2}",
}


def text_profile(df, label):
    """Share of rows showing each noise pattern, per country."""
    name = pl.col("business_name").fill_null("")
    addr = pl.col("business_address").fill_null("")
    aggs = [
        pl.len().alias("rows"),
        name.str.len_chars().mean().round(1).alias("name_len"),
        addr.str.len_chars().mean().round(1).alias("addr_len"),
        ((name == name.str.to_uppercase()) & name.str.contains("[A-Za-z]")).mean().alias("name_all_caps"),
    ]
    aggs += [name.str.contains(p).mean().alias(f"name_{k}") for k, p in NAME_FLAGS.items()]
    aggs += [
        (addr == "").mean().alias("addr_empty"),
        ((addr == addr.str.to_uppercase()) & addr.str.contains("[A-Za-z]")).mean().alias("addr_all_caps"),
        addr.str.contains(r"\p{Devanagari}").mean().alias("addr_devanagari"),
        addr.str.contains(INDIC).mean().alias("addr_other_indic"),
        addr.str.contains(r"^\s*\d").mean().alias("addr_starts_with_number"),
        addr.str.contains(r"\d").mean().alias("addr_has_digit"),
        addr.str.contains(r"\b\d{5}(-\d{4})?\b|\b\d{3}\s?\d{3}\b").mean().alias("addr_5or6_digit_num"),
        addr.str.contains(r"(?i)\b(near|opp|opposite|behind|beside)\b").mean().alias("addr_landmark"),
        (addr.str.count_matches(",") + 1).mean().round(2).alias("addr_n_parts"),
    ]
    return df.group_by("country").agg(aggs).with_columns(file=pl.lit(label))


prof = pl.concat(
    [text_profile(df, f"tr {k}") for k, df in tr.items()] + [text_profile(df, f"te {k}") for k, df in te.items()]
).sort("country", "file")
cols = (prof["country"] + " | " + prof["file"]).to_list()
prof_t = prof.drop("file", "country").transpose(include_header=True, header_name="metric", column_names=cols)
prof_t.with_columns(pl.exclude("metric").round(3))

Read the table column-wise (country | split source). Things to look for: which source uses ALL CAPS addresses, where
non-Latin script appears (names vs addresses), injected accents in US/India (noise), junk prefixes, domain-style names,
and how rarely addresses carry a postcode.

## 4. Name & address vocabulary

In [ ]:
def tokens(col):
    """Lower-cased alphanumeric tokens of a string column."""
    return pl.col(col).fill_null("").str.to_lowercase().str.extract_all(r"[\p{L}\p{M}\p{N}]+")


def top_values(s, k=15):
    """Top-k values of a Series formatted as 'value (share)'."""
    n = len(s)
    vc = s.drop_nulls().value_counts(sort=True).head(k)
    out = [f"{v} ({c / n:.1%})" for v, c in vc.iter_rows()]
    return out + [""] * (k - len(out))


def vocab_table(d, country, k=15):
    """First/last name tokens per source for one country."""
    table = {}
    for pos, label in [(0, "first"), (-1, "last")]:
        for src, df in d.items():
            sub = df.filter(pl.col("country") == country)
            table[f"{src} {label}"] = top_values(sub.select(tokens("business_name").list.get(pos, null_on_oob=True)).to_series(), k)
    return pl.DataFrame(table)


for c in ["US", "India"]:
    print(f"=== train {c}: most common FIRST / LAST name tokens ===")
    display(vocab_table(tr, c))

In [ ]:
def last_segment_table(d, country, k=15):
    """Most common final comma-separated address segment (usually state/region) per source."""
    return pl.DataFrame({
        src: top_values(
            df.filter(pl.col("country") == country)
            .select(pl.col("business_address").str.split(",").list.last().str.strip_chars())
            .to_series(),
            k,
        )
        for src, df in d.items()
    })


for c in ["US", "India"]:
    print(f"=== train {c}: last address segment ===")
    display(last_segment_table(tr, c))

## 5. Example matched clusters

In [ ]:
ATTR = ["entity_id", "business_name", "business_address", "country"]


def show_cluster(s1_id, d=tr):
    """Display one S1 record followed by all of its ground-truth matches."""
    ids = gt.filter(pl.col("source1_entity_id") == s1_id)["ids"][0]
    ids = [] if ids is None else ids.to_list()
    rows = [d["S1"].filter(pl.col("entity_id") == s1_id)]
    rows += [d[k].filter(pl.col("entity_id").is_in(ids)) for k in ("S2", "S3")]
    display(pl.concat(rows).select(ATTR))


for c in ["US", "India"]:
    sample = gt.filter((pl.col("country") == c) & (pl.col("n_match") >= 2)).sample(3, seed=SEED)
    for s1_id in sample["source1_entity_id"]:
        show_cluster(s1_id)

In [ ]:
# Singletons: S1 records with no match at all — do they look different?
display(tr["S1"].join(gt.filter(pl.col("n_match") == 0).select(entity_id="source1_entity_id"), on="entity_id")
        .sample(10, seed=SEED))

## 6. Similarity signal: matched vs random pairs

Sample of true pairs vs. *random same-country* pairs (the match side shuffled within country).
Random negatives are easy — the real negatives after blocking will be much closer — but this shows how noisy the positives are.

In [ ]:
N = 200_000
pos = P.sample(N, seed=SEED)
neg = pl.concat([
    g.with_columns(g.select("name2", "addr2").sample(fraction=1.0, shuffle=True, seed=SEED).get_columns())
    for _, g in pos.group_by("country1", maintain_order=True)
])


def add_sims(df):
    """Add rapidfuzz name/address similarity columns (0-100)."""
    get = lambda c: df[c].fill_null("").to_list()
    n1, n2, a1, a2 = get("name1"), get("name2"), get("addr1"), get("addr2")
    kw = dict(processor=utils.default_process, workers=-1)
    return df.with_columns(
        name_ratio=process.cpdist(n1, n2, scorer=fuzz.ratio, **kw),
        name_token_set=process.cpdist(n1, n2, scorer=fuzz.token_set_ratio, **kw),
        addr_token_set=process.cpdist(a1, a2, scorer=fuzz.token_set_ratio, **kw),
    )


t0 = time.time()
pos, neg = add_sims(pos), add_sims(neg)
print(f"similarities in {time.time() - t0:.0f}s")

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, ["name_ratio", "name_token_set", "addr_token_set"]):
    ax.hist(pos[col], bins=50, alpha=0.6, density=True, label="true pairs")
    ax.hist(neg[col], bins=50, alpha=0.6, density=True, label="random pairs")
    ax.set_title(col); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Where are the hard positives? Split by country, source, and whether the two names use different scripts.
non_latin = r"[\p{Devanagari}\p{Bengali}\p{Gujarati}\p{Gurmukhi}\p{Kannada}\p{Malayalam}\p{Oriya}\p{Tamil}\p{Telugu}]"
pos = pos.with_columns(
    script_mismatch=pl.col("name1").fill_null("").str.contains(non_latin) != pl.col("name2").fill_null("").str.contains(non_latin),
    addr2_missing=pl.col("addr2").is_null(),
)
pos.group_by("country1", "src", "script_mismatch").agg(
    pairs=pl.len(),
    name_ts_median=pl.col("name_token_set").median(),
    name_ts_lt60=(pl.col("name_token_set") < 60).mean().round(3),
    addr_ts_median=pl.col("addr_token_set").median(),
    addr_ts_lt50=(pl.col("addr_token_set") < 50).mean().round(3),
    addr2_missing=pl.col("addr2_missing").mean().round(3),
).sort("country1", "src", "script_mismatch")

In [ ]:
# Hardest Latin-script positives by name: DBA / trade names, acronyms, heavy typos?
(pos.filter(~pl.col("script_mismatch"))
    .sort("name_token_set")
    .select("country1", "src", "name1", "name2", "name_token_set", "addr1", "addr2", "addr_token_set")
    .head(20))

In [ ]:
# Cross-script examples (India): what does transliteration have to solve?
(pos.filter(pl.col("script_mismatch"))
    .sample(15, seed=SEED)
    .select("src", "name1", "name2", "addr1", "addr2"))

## 7. Blocking preview

For the positive sample: how often does a true pair share a cheap key? Name tokens exclude a small list of
legal/generic words. A key with high coverage *and* small blocks is a good blocking key.

In [ ]:
GENERIC = [
    "inc", "llc", "ltd", "pvt", "private", "limited", "corp", "corporation", "co", "company", "the", "and", "of",
    "llp", "pc", "pllc", "lp", "sarl", "sas", "sa", "eurl", "sci", "services", "group", "india", "enterprises",
]


def name_keys(col):
    """Normalised name tokens (len>=2) minus generic/legal words."""
    return (
        pl.col(col).fill_null("").str.to_lowercase().str.extract_all(r"[\p{L}\p{M}\p{N}]{2,}")
        .list.eval(pl.element().filter(~pl.element().is_in(GENERIC)))
    )


house = lambda c: pl.col(c).fill_null("").str.extract(r"(\d+)", 1)
nospace = lambda c: pl.col(c).fill_null("").str.to_lowercase().str.replace_all(r"[^\p{L}\p{N}]", "")

keys = pos.with_columns(
    shares_name_token=name_keys("name1").list.set_intersection(name_keys("name2")).list.len() > 0,
    same_first_number=(house("addr1") == house("addr2")).fill_null(False),
    same_nospace_name=nospace("name1") == nospace("name2"),
)
keys = keys.with_columns(name_or_number=pl.col("shares_name_token") | pl.col("same_first_number"))

keys.group_by("country1", "script_mismatch").agg(
    pairs=pl.len(),
    same_nospace_name=pl.col("same_nospace_name").mean().round(3),
    shares_name_token=pl.col("shares_name_token").mean().round(3),
    same_first_number=pl.col("same_first_number").mean().round(3),
    name_or_number=pl.col("name_or_number").mean().round(3),
).sort("country1", "script_mismatch")

In [ ]:
# Block-size risk: most frequent non-generic name tokens across train S2+S3 (per country)
for c in ["US", "India"]:
    toks = (other_tr.filter(pl.col("country") == c)
            .select(name_keys("business_name").list.unique().alias("t")).explode("t", empty_as_null=True).drop_nulls()["t"])
    vc = toks.value_counts(sort=True)
    print(f"{c}: {vc.height:,} distinct tokens; top-20 (records containing token):")
    display(vc.head(20))

## 8. France (test only)

In [ ]:
for k, df in te.items():
    print(f"=== test {k} France sample ===")
    display(df.filter(pl.col("country") == "France").sample(8, seed=SEED).select(ATTR))

In [ ]:
print("=== test France: most common FIRST / LAST name tokens ===")
display(vocab_table(te, "France"))
print("=== test France: last address segment ===")
display(last_segment_table(te, "France"))

In [ ]:
# Street-type spellings in French addresses (abbreviation variants the normaliser must map)
fr_addr = pl.concat([df.filter(pl.col("country") == "France") for df in te.values()])
street = fr_addr.select(
    pl.col("business_address").fill_null("").str.to_lowercase()
    .str.extract(r"(?:^|,)\s*(?:no|n°|#)?\s*\d+\s*(?:(?:bis|ter|[a-z])\s+)?[-,]?\s*([\p{L}\.]+)", 1).alias("street_type")
)["street_type"]
top_values(street, 25)

## Key takeaways

**Structure**
- Train: 2.2M S1 → 7.64M true pairs. Mean 3.46 matches per S1 (median 3, max 11). Singletons are only **5.6%**.
  80% of entities have both S2 and S3 matches. US and India have almost identical statistics, so the data is synthetic and generated the same way per country.
- **Each S2/S3 record belongs to at most one S1** (0 duplicates), and **true pairs never cross countries**.
  So block within country and assign each S2/S3 record to its best S1 only.
- About **26% of S2/S3 records are distractors** (they match nothing), at the same rate in every source and country.
- Test mix differs from train: India 47% / US 38% / **France 15%** (train is 60/40 US/India).

**Source styles** (S1 is the clean reference)
- S1: no all-caps text, no junk, no Indic script, never an empty address; always ends with a state/region.
- S2: US addresses are about 90% ALL CAPS; uses state codes for the US (TX) but full names for India.
- S3: the reverse. Full state names for the US (Texas), codes for India (MH, DL); Indic-script state names (महाराष्ट्र); `d/b/a` names.
- S2/S3 noise: injected accents (~6% of US names), double spaces (~10%), brackets (~10%), junk prefixes (~2%),
  domain names (~4%), empty addresses (~3%), legal suffixes moved or reordered (`Limited Private`, `PC Genesis,`), prefixes like `Mr/Smt/Dr/Shri`.
- **Postcodes are essentially absent.** Address matching has to rely on house number, street, city and state.
- House-number noise: dropped or added digits (`10108` vs `0108`), leading zeros (`00312`), ranges (`601-605`), suffixes (`5405c`).

**Hard positives**
- About **18% of India pairs are cross-script** (Indic-script name vs Latin): name similarity is about 0 and name tokens
  are shared in only about 6%, but the address still matches well. That calls for transliteration, or a dictionary mined from these pairs.
- About **5% of Latin-script positives have a completely different, invented brand name** (`Evonovi`, `Xylovio`, `Onyxxylo`).
  They are matchable **only by address**, so address must be both a blocking key and a strong feature.
- Otherwise names are close (median token-set similarity 100).

**Blocking preview**
- "Shares a non-generic name token OR same first number" covers about **97–98%** of same-script pairs, but only about 67% of cross-script pairs.
- Frequent tokens (`center`, `com`, `partners`, `care`, `sri`, `shri`…) create huge blocks, so use rare tokens (IDF) or TF-IDF top-K.
- The most frequent India tokens are **Indic-script legal suffixes** (`लिमिटेड`, `प्राइवेट`, `प्रा. लि.`).
  The legal-suffix stripper needs these too, not just `Pvt Ltd`.

**France (test only)**
- Legal forms: SARL / SAS / EURL / SA / SASU / SCI / EI / SNC; `Ets` = `Établissements`; `(France)` inserted.
- S1 ends with a *region* (Hauts-de-France); S2/S3 often use a *département* (Nord, Gironde) or just the city (e.g. LILLE).
- Street-type variants: `Rue / R. / R`, `Avenue / Av. / Av`, `Boulevard / Bd / Bldv`, `Allée / Allee`, plus `N°` / `#` / `bis` / `ter` house numbers.